# Ранжирование русскоязычных поисковых результатов

**AMML ranking competition 2026**  
**Метрика:** NDCG@10  
**Kaggle Public Score:** **0.950200** (финальная модель v9)

**Метод:** listwise-реранкер Jina v3.5, BGE-reranker-v2-m3 с дообучением LoRA и обучаемая модель LightGBM LambdaMART. Обучение проводится только на размеченных парах из конкурсного `train.csv`. Предобученные веса Jina и BGE загружаются из открытых модельных репозиториев.

Эксперименты, анализ данных и инструкции по воспроизведению представлены в `README.md`.


## 0. Окружение и параметры

Используется Kaggle Notebook (Linux, GPU). Первая ячейка устанавливает необходимые NLP-библиотеки, не заменяя предустановленную CUDA-сборку PyTorch.


In [ ]:
# Устанавливаем только NLP-добавки, не переустанавливаем torch из Kaggle GPU!
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'transformers==4.57.1','peft==0.17.1','accelerate==1.12.0',
    'sentencepiece==0.2.1','einops==0.8.1','lightgbm==4.6.0'])

In [ ]:
import os, re, math, random, time, html, unicodedata, platform, gc
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from nltk.stem.snowball import RussianStemmer
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from tqdm.auto import tqdm
import lightgbm as lgb

SEED = 993
MODEL_NAME = 'BAAI/bge-reranker-v2-m3'
VALID_SHARE = 0.12
NUM_EPOCHS = 2
MAX_LENGTH = 320
PAIRS_PER_BATCH = 3      # 3 позитивно-негативные пары = 6 текстов за forward
GRAD_ACCUM = 4
LR = 6e-5
HELDOUT_LR = 8e-6
TRAIN_ON_HELDOUT_AT_END = True  # воспроизводим финальную схему v4; refit не имеет независимой оценки
WEIGHT_DECAY = 0.01
INFER_BATCH = 12
LORA_R = 16

assert torch.cuda.is_available(), 'Требуется GPU: Kaggle -> Settings -> Accelerator'
DEVICE = torch.device('cuda:0')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
print('Platform:',platform.platform(),'Torch:',torch.__version__)
print('GPU:',torch.cuda.get_device_name(0),'GPU total:',torch.cuda.device_count())
JINA_NAME='jinaai/jina-reranker-v3.5'
USE_JINA=True
JINA_GATE=0.0015
JINA_EXTENDED=True         # дополнительное более полное текстовое окно, только если выигрыш на двух половинах
USE_JINA_LTR=True          # supervised LambdaMART поверх Jina и pretrained BGE
LTR_GATE=0.0015            # минимальный прирост на валидации
print('Model:',MODEL_NAME,'Jina:',JINA_NAME)
START_TIME=time.time()
WORK=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
SCRATCH=Path('/kaggle/temp') if Path('/kaggle/temp').exists() else WORK/'scratch'
SCRATCH.mkdir(parents=True,exist_ok=True)
print('Scratch:', SCRATCH, 'Work:', WORK)

# Ранняя проверка конфигурации до скачивания тяжёлых весов.
assert MODEL_NAME and JINA_NAME and isinstance(USE_JINA, bool)
assert 0 < VALID_SHARE < 1 and MAX_LENGTH >= 128 and NUM_EPOCHS >= 1
assert 0 <= JINA_GATE < 1 and PAIRS_PER_BATCH >= 1 and GRAD_ACCUM >= 1
assert isinstance(JINA_EXTENDED,bool) and isinstance(USE_JINA_LTR,bool) and 0<=LTR_GATE<1
print('CONFIG PREFLIGHT OK', flush=True)


## 1. Загрузка данных и метрика

Разделение train/validation выполняется по `query_id`. Метрика NDCG@10 вычисляется по всем кандидатам запроса, включая правило разрешения равных оценок по `id`.


In [ ]:
def find_file(filename):
    roots=[Path.cwd(),Path('/mnt/data')]
    for root in roots:
        p=root/filename
        if p.exists(): return p
    r=Path('/kaggle/input')
    found=sorted(r.rglob(filename)) if r.exists() else []
    if found:return found[0]
    raise FileNotFoundError(f'{filename} not found. Kaggle -> Add Input -> competition.')

train=pd.read_csv(find_file('train.csv'),dtype={'id':str,'query_id':str})
test=pd.read_csv(find_file('test.csv'),dtype={'id':str,'query_id':str})
sample=pd.read_csv(find_file('sample_submission.csv'),dtype={'id':str})
assert {'id','query_id','query','title','text','relevance'}<=set(train.columns)
assert {'id','query_id','query','title','text'}<=set(test.columns)
assert list(sample.columns)==['id','prediction']
assert train.relevance.isin([0,1]).all()
assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
assert set(sample.id)==set(test.id)
assert set(train.query_id).isdisjoint(set(test.query_id))
print('train',train.shape,'queries',train.query_id.nunique(),'positive rate',train.relevance.mean())
print('test ',test.shape,'queries',test.query_id.nunique())
print('missing title',train.title.isna().sum(),'missing passage index',train.passage_index.isna().sum())

splitter=GroupShuffleSplit(n_splits=1,test_size=VALID_SHARE,random_state=SEED)
FIT_IDX,VAL_IDX=next(splitter.split(train,train.relevance,groups=train.query_id))
assert set(train.query_id.iloc[FIT_IDX]).isdisjoint(set(train.query_id.iloc[VAL_IDX]))
print('fit queries',train.query_id.iloc[FIT_IDX].nunique(),
      'validation queries',train.query_id.iloc[VAL_IDX].nunique())

def ndcg10(frame,scores):
    tmp=frame[['id','query_id','relevance']].copy().reset_index(drop=True)
    tmp['score']=np.asarray(scores,dtype=np.float64)
    assert len(tmp)==len(scores) and np.isfinite(tmp.score).all()
    vals=[]
    for _,group in tmp.groupby('query_id',sort=False):
        y=group.relevance.to_numpy(dtype=float)
        sc=group.score.to_numpy(dtype=float)
        ids=group.id.to_numpy(dtype=str)
        order=np.lexsort((ids,-sc)); k=min(10,len(y))
        d=1/np.log2(np.arange(2,k+2))
        dcg=np.sum((2**y[order[:k]]-1)*d)
        ideal=np.sum((2**np.sort(y)[::-1][:k]-1)*d)
        vals.append(dcg/ideal if ideal>0 else 0.)
    return float(np.mean(vals))

_t=pd.DataFrame({'id':['c','a','b'],'query_id':['q']*3,'relevance':[0,1,1]})
assert abs(ndcg10(_t,[0.,2.,1.])-1.)<1e-12


## 2. Представление текстов

Для запроса и фрагмента строятся варианты текста: начало документа и окно с наиболее близкими запросу предложениями.


In [ ]:
STEMMER=RussianStemmer()
WORD_RE=re.compile(r'[a-zа-я0-9]+',re.I)
STOP=set('и в во на не по с со для от у к ко из за об о а но или что это какой какая какие какое кто где когда сколько есть был была было были его ее их'.split())
SENT_RE=re.compile(r'(?<=[.!?])\s+|[\n\r]+')
@lru_cache(maxsize=250000)
def stem_token(w):
    return STEMMER.stem(w) if w.isalpha() and re.fullmatch('[а-я]+',w) else w

def norm(s):
    if not isinstance(s,str):return ''
    s=html.unescape(re.sub('<[^>]+>',' ',s))
    s=unicodedata.normalize('NFC',s).replace('́','')  # убрать только знак ударения, НЕ превращать й в и
    return re.sub(r'\s+',' ',s).strip()

@lru_cache(maxsize=30000)
def query_terms(q):
    words=WORD_RE.findall(norm(q).lower().replace('ё','е'))
    return set(stem_token(w) for w in words if len(w)>2 and w not in STOP)

def window_score(sent,q_terms,q):
    words=[stem_token(w) for w in WORD_RE.findall(sent.lower().replace('ё','е'))]
    if not words:return -1.
    matches=len(set(words)&q_terms)
    score=matches / (math.sqrt(len(words))+1)
    if matches>=2:score+=0.3
    q=q.lower()
    if any(x in q for x in ['когда','каком году','каком веке','в каком году']):
        if re.search(r'(?<!\d)(?:1[0-9]{3}|20[0-9]{2})(?!\d)',sent):score+=0.20
    if any(x in q for x in ['сколько','площадь','высот','вес','длин','населени']):
        if re.search(r'\d',sent):score+=0.13
    return score

def create_views(q,title,passage):
    q=norm(q); title=norm(title); passage=norm(passage)
    header=('Название: '+title+'. ') if title else ''
    # Короткие фрагменты не требуют выбора окна.
    if len(passage)<=900:
        one=header+passage
        return one,one
    first=header+passage[:1150]
    sentences=[x.strip() for x in SENT_RE.split(passage) if len(x.strip())>2]
    if not sentences:
        return first,first
    kw=query_terms(q)
    best=max(range(len(sentences)),key=lambda i:window_score(sentences[i],kw,q))
    left=max(0,best-1);right=min(len(sentences),best+2)
    fragment=' '.join(sentences[left:right])[:980]
    focus=header+passage[:150]+' ... '+fragment
    return first,focus

print('Подготовка двух вариантов текста...')
tr_first=[];tr_focus=[]
for q,t,x in tqdm(zip(train['query'],train['title'],train['text']),total=len(train),desc='Train views'):
    a,b=create_views(q,t,x);tr_first.append(a);tr_focus.append(b)
te_first=[];te_focus=[]
for q,t,x in tqdm(zip(test['query'],test['title'],test['text']),total=len(test),desc='Test views'):
    a,b=create_views(q,t,x);te_first.append(a);te_focus.append(b)
print('Изменившихся train-окон:',sum(a!=b for a,b in zip(tr_first,tr_focus)))
print('Изменившихся test-окон:',sum(a!=b for a,b in zip(te_first,te_focus)))

# Небольшой lexical overlap нужен ТОЛЬКО для выбора сложных негативов, не для прогноза.
def lexical_overlap(q,title,passage):
    ts=query_terms(q)
    if not ts:return 0.
    tt=set(stem_token(t) for t in WORD_RE.findall(norm(title).lower().replace('ё','е')))
    tx=set(stem_token(t) for t in WORD_RE.findall(norm(passage[:1100]).lower().replace('ё','е')))
    return 2.0*len(ts&tt)/len(ts)+len(ts&tx)/len(ts)

hardness=np.asarray([lexical_overlap(q,t,x) for q,t,x in
            zip(train['query'],train['title'],train['text'])],dtype=np.float32)


## 3. Jina v3.5: listwise-оценки

Jina оценивает набор кандидатов внутри одного запроса. На валидации сравниваются порядок документов и альтернативные текстовые представления. При сбое Jina сохраняется возможность обучить BGE.


In [ ]:
import traceback

def jina_doc(q, title, passage):
    """Короткая фактологическая выжимка без внешних данных и без меток."""
    passage = norm(passage)
    title = norm(title)
    header = ("Заголовок: " + title + ". ") if title else ""
    if len(passage) <= 850:
        return (header + passage)[:980]
    _, focus = create_views(q, title, passage)
    relevant = focus.split(' ... ', 1)[-1]
    return (header + "Начало: " + passage[:320] + "\nФрагмент: " + relevant[:570])[:1100]

def jina_doc_expanded(q, title, passage):
    """Второй вариант: больше начала текста + две лучшие неповторяющиеся evidence-области."""
    q=norm(q); title=norm(title); passage=norm(passage)
    header=('Заголовок: '+title+'. ') if title else ''
    if len(passage)<=1450:
        return (header+passage)[:1850]
    sents=[s.strip() for s in SENT_RE.split(passage) if len(s.strip())>2]
    if not sents:
        return (header+passage)[:1850]
    qterms=query_terms(q)
    ordered=sorted(range(len(sents)),key=lambda i:window_score(sents[i],qterms,q),reverse=True)
    chosen=[]
    for i in ordered:
        if all(abs(i-j)>2 for j in chosen):
            chosen.append(i)
        if len(chosen)==2:break
    chunks=[]
    for i in sorted(chosen):
        chunks.append(' '.join(sents[max(0,i-1):min(len(sents),i+2)])[:700])
    return (header+'Начало: '+passage[:610]+'\nОтветные фрагменты: '+' ... '.join(chunks))[:1900]

valid_for_jina = train.iloc[VAL_IDX].reset_index(drop=True)
val_jina_docs = [jina_doc(q,t,x) for q,t,x in zip(valid_for_jina['query'], valid_for_jina['title'], valid_for_jina['text'])]

jina_val = None
jina_test = None
JINA_VIEW = None
JINA_DOC_STYLE = 'compact'
try:
    if USE_JINA:
        print('Загружаем Jina listwise:', JINA_NAME, flush=True)
        # Модель использует собственный remote code и exposes .rerank(query, documents).
        from transformers import AutoModel
        jina_model = AutoModel.from_pretrained(
            JINA_NAME,
            trust_remote_code=True,
            dtype=torch.float16,
        )
        jina_model = jina_model.to(DEVICE).eval()
        assert hasattr(jina_model, 'rerank') and callable(jina_model.rerank), (
            'Jina model loaded but has no .rerank(query, documents) method'
        )

        @torch.inference_mode()
        def jina_by_query(frame, docs, reverse=False):
            scores = np.full(len(frame), np.nan, dtype=np.float32)
            grouped = frame.groupby('query_id', sort=False).indices
            for qid, rows in tqdm(grouped.items(), total=len(grouped), desc='Jina reverse' if reverse else 'Jina listwise'):
                rows = np.asarray(rows, dtype=np.int64)
                order = rows[::-1] if reverse else rows
                query = str(frame.iloc[int(order[0])]['query'])
                candidates = [docs[int(i)] for i in order]
                # Каждый скор в выдаче содержит исходный index в candidates.
                try:
                    answers = jina_model.rerank(query, candidates)
                except torch.cuda.OutOfMemoryError:
                    # В редких длинных группах уменьшаем контекст вместо падения всего обучения.
                    torch.cuda.empty_cache()
                    answers = jina_model.rerank(query, [d[:450] for d in candidates])
                if isinstance(answers, dict) and 'results' in answers:
                    answers = answers['results']
                assert len(answers) == len(order), (qid, len(answers), len(order))
                used = set()
                for answer in answers:
                    pos = int(answer['index'])
                    assert 0 <= pos < len(order) and pos not in used, (qid, pos)
                    used.add(pos)
                    scores[int(order[pos])] = float(answer['relevance_score'])
            assert np.isfinite(scores).all(), 'Jina returned NaN or missing result'
            return scores

        jina_val_a = jina_by_query(valid_for_jina, val_jina_docs, reverse=False)
        jina_val_b = jina_by_query(valid_for_jina, val_jina_docs, reverse=True)
        jina_variants = {
            'original': jina_val_a,
            'reverse': jina_val_b,
            'two_orders': (jina_val_a + jina_val_b) / 2,
        }
        jina_quality = {k: ndcg10(valid_for_jina, v) for k,v in jina_variants.items()}
        JINA_VIEW = max(jina_quality, key=jina_quality.get)
        jina_val = jina_variants[JINA_VIEW]
        print('JINA VALIDATION:', {k: round(v,6) for k,v in jina_quality.items()}, 'chosen', JINA_VIEW,flush=True)

        # Добавление расширенного документа контролируем на двух независимых половинах запросов.
        if JINA_EXTENDED:
            try:
                val_ext_docs=[jina_doc_expanded(q,t,x) for q,t,x in zip(valid_for_jina['query'],valid_for_jina['title'],valid_for_jina['text'])]
                ext_score=jina_by_query(valid_for_jina,val_ext_docs,reverse=False)
                names=np.asarray(sorted(valid_for_jina.query_id.unique()))
                np.random.default_rng(2026).shuffle(names)
                left=set(names[:len(names)//2])
                m=valid_for_jina.query_id.isin(left).to_numpy()
                compact_halves=(ndcg10(valid_for_jina.loc[m],jina_val[m]),ndcg10(valid_for_jina.loc[~m],jina_val[~m]))
                ext_halves=(ndcg10(valid_for_jina.loc[m],ext_score[m]),ndcg10(valid_for_jina.loc[~m],ext_score[~m]))
                print('JINA EXTENDED SCOUT: compact',round(ndcg10(valid_for_jina,jina_val),6),
                      'expanded',round(ndcg10(valid_for_jina,ext_score),6),
                      'halves compact',tuple(round(z,6) for z in compact_halves),
                      'expanded',tuple(round(z,6) for z in ext_halves),flush=True)
                if (ndcg10(valid_for_jina,ext_score)>ndcg10(valid_for_jina,jina_val)+0.0015
                    and ext_halves[0]>compact_halves[0]+0.0003
                    and ext_halves[1]>compact_halves[1]+0.0003):
                    jina_val=ext_score
                    JINA_DOC_STYLE='expanded'
                    JINA_VIEW='original'
                    print('EXPANDED DOC STYLE SELECTED',flush=True)

            except Exception as ext_error:
                print('JINA EXPANDED SCOUT FAILED; compact Jina retained:',repr(ext_error),flush=True)

        print('JINA SCOUT READY: test inference is deferred until BGE selection',flush=True)
except Exception as err:
    print('Jina is unavailable; supervised BGE fallback will continue.',repr(err),flush=True)
    traceback.print_exc(limit=3)
    jina_val = jina_test = None
    JINA_VIEW = None
finally:
    if 'jina_model' in globals():
        del jina_model
    gc.collect()
    torch.cuda.empty_cache()

## 4. BGE-reranker-v2-m3 и LoRA

Загружаются исходные веса BGE. Модель дообучается далее на парах релевантных и нерелевантных кандидатов из конкурсных данных.


In [ ]:
print('Загружаем токенизатор и веса...')
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
base=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
assert base.config.num_labels==1,'Ожидается один логит релевантности'
# В XLM-R attention есть модули query и value.
assert any(n.endswith('.query') for n,_ in base.named_modules()),'Не найдены attention.query'
assert any(n.endswith('.value') for n,_ in base.named_modules()),'Не найдены attention.value'
lora=LoraConfig(task_type=TaskType.SEQ_CLS,r=LORA_R,lora_alpha=LORA_R*2,
                lora_dropout=0.05,target_modules=['query','value'],
                modules_to_save=['classifier'],bias='none')
model=get_peft_model(base,lora)
model.to(DEVICE)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
model.print_trainable_parameters()

def encode_views(frame,first,focus):
    queries=frame['query'].fillna('').astype(str).tolist()
    def encode_docs(docs):
        out=[]
        for start in tqdm(range(0,len(docs),2000),desc='Tokenization',leave=False):
            a=tokenizer(queries[start:start+2000],docs[start:start+2000],
                truncation='only_second',max_length=MAX_LENGTH,padding=False,
                return_token_type_ids=False)
            out.extend([{key:value[i] for key,value in a.items()}
                        for i in range(len(docs[start:start+2000]))])
        return out
    enc_a=encode_docs(first)
    enc_b=enc_a.copy()
    changed=[i for i,(a,b) in enumerate(zip(first,focus)) if a!=b]
    for start in tqdm(range(0,len(changed),1000),desc='Changed views',leave=False):
        ids=changed[start:start+1000]
        a=tokenizer([queries[i] for i in ids],[focus[i] for i in ids],
                    truncation='only_second',max_length=MAX_LENGTH,padding=False,
                    return_token_type_ids=False)
        for j,i in enumerate(ids):
            enc_b[i]={key:value[j] for key,value in a.items()}
    return enc_a,enc_b,np.asarray(changed,dtype=np.int64)

tr_a,tr_b,tr_changed=encode_views(train,tr_first,tr_focus)
te_a,te_b,te_changed=encode_views(test,te_first,te_focus)
print('Tokenization done',len(tr_a),len(te_a))


## 5. Базовая оценка BGE

До дообучения вычисляются оценки исходной BGE для сравнения и признаков второй ступени.


In [ ]:
@torch.inference_mode()
def predict_tokens(model,tokenized,ids,batch_size=INFER_BATCH):
    model.eval();ids=np.asarray(ids,dtype=int)
    output=np.empty(len(ids),dtype=np.float32)
    for start in tqdm(range(0,len(ids),batch_size),desc='Predict',leave=False):
        ix=ids[start:start+batch_size]
        batch=tokenizer.pad([tokenized[int(i)] for i in ix],padding=True,
                           pad_to_multiple_of=8,return_tensors='pt')
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        with torch.autocast('cuda',dtype=torch.float16):
            sc=model(**batch).logits.flatten().float()
        output[start:start+len(ix)]=sc.cpu().numpy()
    assert np.isfinite(output).all()
    return output

def predict_two_views(model,enc_a,enc_b,changed,ids):
    ids=np.asarray(ids,dtype=np.int64)
    a=predict_tokens(model,enc_a,ids)
    b=a.copy()
    # Предсказываем второй вариант только у длинных фрагментов с другим окном.
    mask=np.isin(ids,changed)
    if mask.any():b[mask]=predict_tokens(model,enc_b,ids[mask])
    return a,b

def view_variants(a,b):
    return {'first':a,'focus':b,'mean':(a+b)/2,'max':np.maximum(a,b)}

def choose_view(frame,a,b,name):
    scores={k:ndcg10(frame,v) for k,v in view_variants(a,b).items()}
    print(name,{k:round(v,6) for k,v in scores.items()},flush=True)
    winner=max(scores,key=scores.get)
    return winner,scores[winner]

valid=train.iloc[VAL_IDX].reset_index(drop=True)
print('Zero-shot validation:')
zero_val_a,zero_val_b=predict_two_views(model,tr_a,tr_b,tr_changed,VAL_IDX)
ZERO_VIEW,ZERO_SCORE=choose_view(valid,zero_val_a,zero_val_b,'Pretrained')
# Zero-shot на test вычисляем один раз, пока адаптеры ещё не обучались.
zero_test_a,zero_test_b=predict_two_views(model,te_a,te_b,te_changed,np.arange(len(test)))

# Для второй ступени получаем out-of-training-like сигналы от исходной BGE:
# эти значения вычислены ДО дообучения BGE и поэтому не переобучены под FIT-метки.
if USE_JINA_LTR and jina_val is not None:
    print('Pretrained BGE scores on FIT for supervised listwise calibrator',flush=True)
    zero_fit_a,zero_fit_b=predict_two_views(model,tr_a,tr_b,tr_changed,FIT_IDX)


## 6. Дообучение BGE

Парная функция потерь RankNet, дополненная pointwise-компонентой. Выбор эпохи выполняется по NDCG@10 на запросах validation.


In [ ]:
# Группы train и положительные/отрицательные кандидаты
Y=train.relevance.to_numpy(dtype=np.int8)
GROUPS={qid:np.asarray(idx,dtype=np.int64) for qid,idx in train.groupby('query_id',sort=False).indices.items()}
FIT_Q=train.query_id.iloc[FIT_IDX].drop_duplicates().tolist()
VAL_Q=train.query_id.iloc[VAL_IDX].drop_duplicates().tolist()

def make_pairs(keys,seed):
    rng=np.random.default_rng(seed);pairs=[]
    for qid in rng.permutation(keys):
        rows=GROUPS[qid]
        pos=rows[Y[rows]==1];neg=rows[Y[rows]==0]
        if not len(pos) or not len(neg): continue
        p=int(rng.choice(pos))
        if rng.random()<0.75:
            order=neg[np.argsort(-hardness[neg],kind='stable')]
            n=int(rng.choice(order[:min(4,len(order))]))
        else:
            n=int(rng.choice(neg))
        pairs.append((p,n))
    rng.shuffle(pairs)
    return pairs

def iter_pair_batches(pairs,seed):
    rng=np.random.default_rng(seed)
    for offset in range(0,len(pairs),PAIRS_PER_BATCH):
        chunk=pairs[offset:offset+PAIRS_PER_BATCH]
        rows=[p for p,n in chunk]+[n for p,n in chunk]
        views=[tr_a[i] if rng.random()<0.5 else tr_b[i] for i in rows]
        batch=tokenizer.pad(views,padding=True,pad_to_multiple_of=8,return_tensors='pt')
        yield {k:v.to(DEVICE) for k,v in batch.items()},len(chunk)

def epoch_train(keys,seed,optimizer,scheduler=None):
    model.train();pairs=make_pairs(keys,seed)
    scaler=torch.amp.GradScaler('cuda')
    optimizer.zero_grad(set_to_none=True)
    losses=[]
    nbatches=math.ceil(len(pairs)/PAIRS_PER_BATCH)
    for step,(batch,npairs) in enumerate(tqdm(iter_pair_batches(pairs,seed),
                                              total=nbatches,desc='RankNet training')):
        with torch.autocast('cuda',dtype=torch.float16):
            logits=model(**batch).logits.flatten().float()
            sp,sn=logits[:npairs],logits[npairs:]
            loss=F.softplus(-(sp-sn)).mean()
            # Умеренная pointwise-компонента для калибровки логитов
            loss=loss+0.10*(F.softplus(-sp).mean()+F.softplus(sn).mean())*0.5
        if not torch.isfinite(loss):raise RuntimeError(f'NaN/Inf train loss at step {step}')
        scaler.scale(loss/GRAD_ACCUM).backward()
        losses.append(float(loss.detach().cpu()))
        if (step+1)%GRAD_ACCUM==0 or (step+1)==nbatches:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(optimizer);scaler.update()
            if scheduler is not None:scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    return float(np.mean(losses)),len(pairs)

num_batches=math.ceil(len(FIT_Q)/PAIRS_PER_BATCH)
total_updates=math.ceil(num_batches/GRAD_ACCUM)*NUM_EPOCHS
optimizer=AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=get_linear_schedule_with_warmup(optimizer,
                 num_warmup_steps=max(1,int(total_updates*0.1)),num_training_steps=total_updates)
results=[];best_ft=-float('inf');best_epoch=None
checkpoint=SCRATCH/'best_bge_lora_state.pt'
for ep in range(1,NUM_EPOCHS+1):
    avg,n_pairs=epoch_train(FIT_Q,SEED+ep,optimizer,scheduler)
    va,vb=predict_two_views(model,tr_a,tr_b,tr_changed,VAL_IDX)
    selected,score=choose_view(valid,va,vb,f'Finetuned epoch={ep}')
    results.append({'epoch':ep,'pair_loss':avg,'pairs':n_pairs,
                    'valid_ndcg10':score,'window_mode':selected})
    print(f'EP {ep}: pair loss {avg:.4f}, valid NDCG@10 {score:.6f}')
    if score>best_ft:
        best_ft=score;best_epoch=ep;FT_VIEW=selected
        best_val_a,best_val_b=va.copy(),vb.copy()
        torch.save({k:v.detach().cpu().clone() for k,v in
                    get_peft_model_state_dict(model).items()},checkpoint)
print(pd.DataFrame(results).to_string(index=False))
print('Best fine-tuned epoch:',best_epoch,'NDCG@10:',best_ft,'view:',FT_VIEW)
assert checkpoint.is_file()
set_peft_model_state_dict(model,torch.load(checkpoint,map_location='cpu',weights_only=True))
model.to(DEVICE);model.eval()


## 7. Валидация и выбор базовых оценок

Сравниваются результаты BGE и Jina на непересекающихся с обучением запросах.


In [ ]:
def query_standardize(frame, raw):
    z=np.asarray(raw,dtype=np.float64)
    out=np.empty(len(z), dtype=np.float64)
    for _,idx in frame.groupby('query_id',sort=False).indices.items():
        idx=np.asarray(idx,dtype=int)
        x=z[idx]; sd=x.std()
        out[idx]=(x-x.mean())/max(sd,0.3)
    return np.clip(out,-8.,8.)

zero_chosen=view_variants(zero_val_a,zero_val_b)[ZERO_VIEW]
ft_chosen=view_variants(best_val_a,best_val_b)[FT_VIEW]
base_std=query_standardize(valid,zero_chosen)
ft_std=query_standardize(valid,ft_chosen)
base_candidates=[]
for alpha in [0.25,0.4,0.55,0.7,0.85,1.0]:
    candidate=alpha*ft_std+(1-alpha)*base_std
    score=ndcg10(valid,candidate)
    base_candidates.append((alpha,score))
BEST_ALPHA, BGE_VAL_SCORE=max(base_candidates,key=lambda r:r[1])
base_valid=BEST_ALPHA*ft_std+(1.-BEST_ALPHA)*base_std
print('BGE pretrain:',round(ZERO_SCORE,6),'BGE fine-tuned:',round(best_ft,6))
print('BGE blend alpha',BEST_ALPHA,'val NDCG@10',round(BGE_VAL_SCORE,6))

# Two-way group holdout for selection robustness. Do not look at Kaggle public score.
qids=np.asarray(valid.query_id.drop_duplicates().tolist())
rng=np.random.default_rng(2026);rng.shuffle(qids)
q_left=set(qids[:len(qids)//2]);q_right=set(qids[len(qids)//2:])
mask_left=valid.query_id.isin(q_left).to_numpy()
mask_right=valid.query_id.isin(q_right).to_numpy()
assert mask_left.any() and mask_right.any() and not np.any(mask_left & mask_right)

def score_halves(x):
    return (ndcg10(valid.loc[mask_left],np.asarray(x)[mask_left]),
            ndcg10(valid.loc[mask_right],np.asarray(x)[mask_right]))

bge_halves=score_halves(base_valid)
JINA_WEIGHT=0.0
if jina_val is not None:
    jina_std=query_standardize(valid,jina_val)
    print('JINA standalone valid NDCG:',round(ndcg10(valid,jina_std),6))
    print('BGE halves:',tuple(round(x,6) for x in bge_halves))
    candidates=[]
    for weight in [0.0,0.15,0.3,0.45,0.6,0.75,1.0]:
        candidate=(1-weight)*base_valid+weight*jina_std
        s=ndcg10(valid,candidate)
        h1,h2=score_halves(candidate)
        candidates.append((weight,s,h1,h2))
    print('JINA BLEND VALIDATION (weight, all, left, right):')
    for w,s,h1,h2 in candidates:
        print(f'  {w:.2f}  {s:.6f}  {h1:.6f}  {h2:.6f}')
    # Do not switch because of a single lucky split; require meaningful gain in both halves.
    eligible=[row for row in candidates if
              row[0]>0 and row[1]>BGE_VAL_SCORE+JINA_GATE and
              row[2]>bge_halves[0]+0.0005 and row[3]>bge_halves[1]+0.0005]
    if eligible:
        JINA_WEIGHT=max(eligible,key=lambda v:v[1])[0]
else:
    print('JINA unavailable. Selecting BGE-only.')
print('FINAL CHOICE: JINA_WEIGHT=',JINA_WEIGHT,
      'BGE_WEIGHT=',round(1.-JINA_WEIGHT,3), 'BGE alpha',BEST_ALPHA,flush=True)

## 8. LambdaMART поверх оценок Jina и BGE

LightGBMRanker обучается на 48 признаках, включая скоры и ранги Jina/BGE, пересечения слов, чисел и дат и метаданные статьи. Второй этап проверяется на отложенных запросах.


In [ ]:
# ======================================
# SUPERVISED JINA → LAMBDAMART
# ======================================
import traceback

NUM_RE_LTR=re.compile(r'\d+(?:[.,]\d+)?')
YEAR_RE_LTR=re.compile(r'(?<!\d)(?:1\d{3}|20\d{2})(?!\d)')

def parse_numeric_ltr(x):
    if pd.isna(x): return -1.
    s=str(x).strip().replace('\xa0','').replace(' ','')
    try:
        return float(s.replace(',','')) if s.count(',')>1 else float(s.replace(',','.'))
    except ValueError:
        return -1.

def supervised_features(frame, jina_scores, bge_first, bge_focus):
    """Jina teacher scores + query-relative + lexical/factual + article metadata.
    No labels, external corpus, test statistics fitted globally, nor raw IDs.
    """
    frame=frame.reset_index(drop=True)
    assert len(frame)==len(jina_scores)==len(bge_first)==len(bge_focus)
    j=np.asarray(jina_scores,dtype=np.float64)
    a=np.asarray(bge_first,dtype=np.float64)
    b=np.asarray(bge_focus,dtype=np.float64)
    assert np.isfinite(j).all() and np.isfinite(a).all() and np.isfinite(b).all()
    rows=[]
    for q,t,txt,pidx,pcount,chars,src,js,bs1,bs2 in tqdm(zip(
          frame['query'],frame['title'],frame['text'],frame['passage_index'],
          frame['article_passage_count'],frame['article_char_count'],
          frame['ingest_source'],j,a,b),total=len(frame),desc='Supervised LTR features',leave=False):
        q=norm(q);t=norm(t);txt=norm(txt)
        qs=query_terms(q)
        tt=set(stem_token(w) for w in WORD_RE.findall(t.lower().replace('ё','е')))
        dt=set(stem_token(w) for w in WORD_RE.findall(txt.lower().replace('ё','е')))
        first=set(stem_token(w) for w in WORD_RE.findall(txt[:280].lower().replace('ё','е')))
        qden=max(1,len(qs)); qnum=set(NUM_RE_LTR.findall(q))
        tnums=set(NUM_RE_LTR.findall(t)); dnums=set(NUM_RE_LTR.findall(txt))
        src=norm(src).lower()
        rows.append({
          'jina_raw':float(js), 'bge_first':float(bs1), 'bge_focus':float(bs2),
          'bge_mean':float((bs1+bs2)/2), 'bge_max':float(max(bs1,bs2)),
          'bge_absdiff':float(abs(bs1-bs2)),
          'title_overlap':len(qs&tt)/qden,'doc_overlap':len(qs&dt)/qden,
          'doc_start_overlap':len(qs&first)/qden,
          'title_jaccard':len(qs&tt)/max(1,len(qs|tt)),
          'doc_jaccard':len(qs&dt)/max(1,len(qs|dt)),
          'phrase_in_title':float(bool(q) and q.lower().strip(' ?!') in t.lower()),
          'numeric_query':float(bool(qnum)),
          'number_title_hit':len(qnum&tnums)/max(1,len(qnum)),
          'number_text_hit':len(qnum&dnums)/max(1,len(qnum)),
          'year_in_doc':float(bool(YEAR_RE_LTR.search(txt))),
          'year_in_title':float(bool(YEAR_RE_LTR.search(t))),
          'q_when':float(any(w in q.lower() for w in ('когда','году','веке','дата'))),
          'q_where':float(any(w in q.lower() for w in ('где','какой стране','каком городе'))),
          'q_how_many':float(any(w in q.lower() for w in ('сколько','количество','число'))),
          'q_who':float(any(w in q.lower() for w in ('кто','кем','чьё','чей'))),
          'title_len':np.log1p(len(t)), 'text_len':np.log1p(len(txt)),
          'query_len':np.log1p(len(q)),
          'title_words':len(tt),'doc_words':len(dt),
          'passage_index':parse_numeric_ltr(pidx),
          'passage_count':parse_numeric_ltr(pcount),
          'article_char_count':parse_numeric_ltr(chars),
          'source_wiki':float('wiki' in src),
          'source_html':float('html' in src),
          'source_web':float('web' in src),
          'title_missing':float(not bool(t)),
        })
    X=pd.DataFrame(rows,dtype=np.float32)
    group=frame.query_id.reset_index(drop=True)
    for col in ('jina_raw','bge_first','bge_focus','bge_max','title_overlap','doc_overlap'):
        g=X[col].groupby(group,sort=False)
        mean=g.transform('mean')
        std=g.transform('std').fillna(0.)
        X[col+'_relmean']=(X[col]-mean).astype(np.float32)
        X[col+'_z']=((X[col]-mean)/std.clip(lower=0.25)).astype(np.float32)
    # Jina rank and gap are especially useful for query-relative decisions.
    X['jina_rank']=X['jina_raw'].groupby(group).rank(method='average',ascending=False).astype(np.float32)
    X['jina_gap_to_best']=(X['jina_raw'].groupby(group).transform('max')-X['jina_raw']).astype(np.float32)
    X['group_count']=group.map(group.value_counts()).to_numpy(dtype=np.float32)
    X=X.replace([np.inf,-np.inf],np.nan).fillna(-1.).astype(np.float32)
    assert not X.isna().any().any() and np.isfinite(X.to_numpy()).all()
    return X

def query_sorted_indices(frame):
    """Порядок строк и группы для Lambdarank (не смешиваем запросы)."""
    order=np.argsort(frame.query_id.to_numpy(),kind="stable")
    keys=frame.query_id.to_numpy()[order]
    cuts=np.r_[0,np.flatnonzero(keys[1:]!=keys[:-1])+1,len(keys)]
    groups=np.diff(cuts).astype(int).tolist()
    assert sum(groups)==len(frame) and min(groups)>0
    return order, groups


LTR_WEIGHT=0.0
LTR_RANKER=None
LTR_COLUMNS=None
LTR_JINA_FIT=None
JINA_LTR_QUALITY=None
VALID_SELECTED=(1.-JINA_WEIGHT)*base_valid
if jina_val is not None:
    VALID_SELECTED=VALID_SELECTED+JINA_WEIGHT*query_standardize(valid,jina_val)
BASELINE_VALID_SCORE=ndcg10(valid,VALID_SELECTED)
base_halves=score_halves(VALID_SELECTED)
print('V8.1 BASELINE REPRODUCTION (before holdout refit):',round(BASELINE_VALID_SCORE,6),
      'two halves:',tuple(round(x,6) for x in base_halves),flush=True)

if USE_JINA_LTR and jina_val is not None:
    try:
        # Important: Jina is NOT trained; supervised LGBM sees only FIT labels.
        fit_frame=train.iloc[FIT_IDX].reset_index(drop=True)
        assert len(zero_fit_a)==len(fit_frame)
        print('Freeing BGE GPU before Jina teacher inference on FIT queries',flush=True)
        del optimizer,scheduler
        model.to('cpu')
        gc.collect();torch.cuda.empty_cache()
        jina_model=AutoModel.from_pretrained(JINA_NAME,trust_remote_code=True,dtype=torch.float16)
        jina_model=jina_model.to(DEVICE).eval()
        assert hasattr(jina_model,'rerank')
        if JINA_DOC_STYLE=='expanded':
            train_docs=[jina_doc_expanded(q,t,x) for q,t,x in zip(
                fit_frame['query'],fit_frame['title'],fit_frame['text'])]
        else:
            train_docs=[jina_doc(q,t,x) for q,t,x in zip(
                fit_frame['query'],fit_frame['title'],fit_frame['text'])]
        if JINA_VIEW=='reverse':
            LTR_JINA_FIT=jina_by_query(fit_frame,train_docs,reverse=True)
        else:
            LTR_JINA_FIT=jina_by_query(fit_frame,train_docs,reverse=False)
            if JINA_VIEW=='two_orders':
                LTR_JINA_FIT=(LTR_JINA_FIT+jina_by_query(fit_frame,train_docs,reverse=True))/2
        del jina_model
        gc.collect();torch.cuda.empty_cache()
        model.to(DEVICE);model.eval()
        # Leakage guard: the fit Jina scores are computed without the labels.
        X_fit=supervised_features(fit_frame,LTR_JINA_FIT,zero_fit_a,zero_fit_b)
        X_valid=supervised_features(valid,jina_val,zero_val_a,zero_val_b)
        assert X_fit.columns.equals(X_valid.columns)
        LTR_COLUMNS=X_fit.columns.tolist()
        fit_order,fit_groups=query_sorted_indices(fit_frame)
        val_order,val_groups=query_sorted_indices(valid)
        trained=[]
        for leaves in (7,15):
            learner=lgb.LGBMRanker(
                objective='lambdarank',metric='ndcg',eval_at=[10],
                n_estimators=700,learning_rate=0.025,num_leaves=leaves,
                min_child_samples=28 if leaves==7 else 35,
                colsample_bytree=0.9,reg_lambda=5.0 if leaves==7 else 8.0,
                lambdarank_truncation_level=10,
                random_state=SEED,n_jobs=4,verbosity=-1)
            learner.fit(X_fit.iloc[fit_order],fit_frame.relevance.iloc[fit_order].astype(int),
                group=fit_groups,
                eval_set=[(X_valid.iloc[val_order],valid.relevance.iloc[val_order].astype(int))],
                eval_group=[val_groups],eval_at=[10],
                callbacks=[lgb.early_stopping(75,verbose=False)])
            ltr_val=learner.predict(X_valid,num_iteration=learner.best_iteration_)
            s=ndcg10(valid,ltr_val)
            print('SUPERVISED JINA LTR',leaves,'leaves',
                  'iterations',learner.best_iteration_,'validation',round(s,6),flush=True)
            trained.append((s,leaves,learner,ltr_val))
        JINA_LTR_QUALITY,chosen_leaves,LTR_RANKER,ltr_val=max(trained,key=lambda z:z[0])
        ltr_std=query_standardize(valid,ltr_val)
        options=[]
        for w in (0.0,0.15,0.3,0.45,0.6,0.75,1.0):
            blend=(1-w)*VALID_SELECTED+w*ltr_std
            score=ndcg10(valid,blend)
            h1,h2=score_halves(blend)
            options.append((w,score,h1,h2))
        print('SUPERVISED ENSEMBLE CANDIDATES (LTR_WEIGHT, overall, half1, half2):')
        for w,score,h1,h2 in options:
            print(f'  {w:.2f} {score:.6f} {h1:.6f} {h2:.6f}')
        eligible=[row for row in options if row[0]>0 and
                  row[1]>BASELINE_VALID_SCORE+LTR_GATE and
                  row[2]>=base_halves[0]-0.0005 and
                  row[3]>=base_halves[1]-0.0005]
        if eligible:
            LTR_WEIGHT=max(eligible,key=lambda row:row[1])[0]
        print('LTR SELECTED WEIGHT:',LTR_WEIGHT,'BGE/Jina baseline:',round(BASELINE_VALID_SCORE,6),flush=True)
        del X_fit,X_valid,fit_frame,train_docs
    except Exception as err:
        print('SUPERVISED JINA LTR FAILED; v8.1 fallback:',repr(err),flush=True)
        traceback.print_exc(limit=3)
        LTR_WEIGHT=0.0
        LTR_RANKER=None
    finally:
        if 'jina_model' in globals():del jina_model
        gc.collect();torch.cuda.empty_cache()
        model.to(DEVICE);model.eval()
else:
    print('Supervised Jina LTR skipped (no Jina validation).',flush=True)


## 9. Финальное обучение BGE и прогноз

После выбора конфигурации BGE выполняется дополнительное обучение на ранее отложенной части `train.csv`. Это использует доступную разметку, но обновлённое состояние не имеет отдельной независимой валидации.


In [ ]:
if TRAIN_ON_HELDOUT_AT_END:
    print('Короткое финальное обучение на отложенных запросах (после выбора по validation).')
    final_optimizer=AdamW([p for p in model.parameters() if p.requires_grad],
                          lr=HELDOUT_LR,weight_decay=WEIGHT_DECAY)
    final_loss,num_extra=epoch_train(VAL_Q,SEED+10000,final_optimizer,None)
    print('Final extra pairs:',num_extra,'loss:',round(final_loss,5))
    del final_optimizer
    gc.collect();torch.cuda.empty_cache()

model.eval()
ft_test_a,ft_test_b=predict_two_views(model,te_a,te_b,te_changed,np.arange(len(test)))
zero_score_test=view_variants(zero_test_a,zero_test_b)[ZERO_VIEW]
ft_score_test=view_variants(ft_test_a,ft_test_b)[FT_VIEW]
bge_test = BEST_ALPHA*query_standardize(test,ft_score_test)+(1.-BEST_ALPHA)*query_standardize(test,zero_score_test)
model.save_pretrained(str(SCRATCH/'final_bge_lora_adapter'))
print('BGE TEST ready. Jina will only run if it improved validation.')


## 10. Расчёт оценок тестовых пар

Выбранная комбинация моделей применяется ко всем исходным кандидатам `test.csv`.


In [ ]:
final_score=bge_test.copy()
if (JINA_WEIGHT>0 or LTR_WEIGHT>0) and jina_val is not None:
    print('Jina confirmed on validation: calculating TEST listwise scores.',flush=True)
    # Free BGE GPU memory before loading Jina.
    if 'model' in globals(): del model
    if 'base' in globals(): del base
    gc.collect();torch.cuda.empty_cache()
    try:
        jina_model=AutoModel.from_pretrained(JINA_NAME,trust_remote_code=True,dtype=torch.float16)
        jina_model=jina_model.to(DEVICE).eval()
        test_jina_builder = jina_doc_expanded if JINA_DOC_STYLE=='expanded' else jina_doc
        test_jina_docs=[test_jina_builder(q,t,x) for q,t,x in zip(test['query'],test['title'],test['text'])]
        jina_test_a=jina_by_query(test,test_jina_docs,reverse=False)
        if JINA_VIEW=='original':
            jina_test=jina_test_a
        else:
            jina_test_b=jina_by_query(test,test_jina_docs,reverse=True)
            jina_test=jina_test_b if JINA_VIEW=='reverse' else (jina_test_a+jina_test_b)/2
        final_score=(1.-JINA_WEIGHT)*bge_test+JINA_WEIGHT*query_standardize(test,jina_test)
        if LTR_WEIGHT>0:
            X_test=supervised_features(test,jina_test,zero_test_a,zero_test_b)
            assert LTR_RANKER is not None and X_test.columns.tolist()==LTR_COLUMNS
            ltr_test=LTR_RANKER.predict(X_test,num_iteration=LTR_RANKER.best_iteration_)
            assert np.isfinite(ltr_test).all()
            final_score=(1.-LTR_WEIGHT)*final_score+LTR_WEIGHT*query_standardize(test,ltr_test)
            print('SUPERVISED JINA LTR ACTIVE. Weight:',LTR_WEIGHT,flush=True)
        print('JINA TEST successful; ensemble active.',flush=True)
    except Exception as err:
        print('Jina TEST failed, keeping fully trained BGE instead.',repr(err),flush=True)
        traceback.print_exc(limit=3)
        final_score=bge_test.copy()
    finally:
        if 'jina_model' in globals(): del jina_model
        gc.collect();torch.cuda.empty_cache()
else:
    print('Jina not confirmed; skipping expensive TEST Jina run, use BGE only.',flush=True)

## 11. Создание файла отправки

Создаётся `/kaggle/working/submission.csv` с колонками `id,prediction` в порядке `sample_submission.csv`.


In [ ]:
assert np.isfinite(final_score).all()
assert np.unique(final_score).size>100

by_id=pd.Series(final_score,index=test.id.to_numpy())
submission=sample[['id']].copy()
submission['prediction']=submission.id.map(by_id)
assert submission.columns.tolist()==['id','prediction']
assert len(submission)==len(test)
assert submission.prediction.notna().all()
assert submission.id.equals(sample.id)
assert np.isfinite(submission.prediction.to_numpy()).all()

outpath=WORK/'submission.csv'
submission.to_csv(outpath,index=False)
check=pd.read_csv(outpath,dtype={'id':str})
assert len(check)==len(sample) and check.id.tolist()==sample.id.tolist()
assert list(check.columns)==['id','prediction']
assert check.prediction.notna().all()
print('SUBMISSION READY:',outpath,'rows:',len(submission))
print('Local BGE validation NDCG@10 BEFORE holdout refit:',round(BGE_VAL_SCORE,6))
print('JINA validation-selected weight:',JINA_WEIGHT)
print('SUPERVISED JINA LTR weight:',LTR_WEIGHT)
print('Elapsed seconds:',round(time.time()-START_TIME))
print(submission.head(5).to_string(index=False))
# Сохраняем АДАПТЕР, а не полную копию pretrained весов.


print('FINAL RUN COMPLETED',flush=True)
